In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

c:\Users\loaye\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\loaye\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
def create_features(df):
    df = df.copy()

    df['Hour'] = (df['Time'] // 3600).astype(int) % 24
    df['Day'] = (df['Time'] // (3600 * 24)).astype(int) % 7

    df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    
    conditions = [
        (df['Hour'] >= 6) & (df['Hour'] < 12),
        (df['Hour'] >= 12) & (df['Hour'] < 18),
        (df['Hour'] >= 18) & (df['Hour'] < 24),
    ]
    choices = ['Morning', 'Afternoon', 'Evening']
    df['Time_of_day'] = np.select(conditions, choices, default='Night')
    
    df["amount_log"] = np.log1p(df["Amount"])
    return df

In [3]:
train_df = create_features(train_df)
val_df = create_features(val_df)
test_df = create_features(test_df)

In [4]:
print(train_df['Day'].value_counts())

Day
0    144236
1     54372
Name: count, dtype: int64


Dataset only has records for Sunday (0) and Monday (1)

In [5]:
train_df[['Time', 'Hour', 'Day', 'Time_of_day', 'Class']].sample(n=5, random_state=42)

,Time,Hour,Day,Time_of_day,Class
53249,46005.0,12,0,Afternoon,0
39511,39925.0,11,0,Morning,0
94412,65022.0,18,0,Evening,0
158479,112191.0,7,1,Morning,0
149059,91604.0,1,1,Night,0


In [6]:
val_df[['Time', 'Hour', 'Day', 'Time_of_day', 'Class']].sample(n=5, random_state=42)

,Time,Hour,Day,Time_of_day,Class
24664,143651.0,15,1,Afternoon,0
37473,149036.0,17,1,Afternoon,0
37524,149055.0,17,1,Afternoon,0
24202,143463.0,15,1,Afternoon,0
38485,149476.0,17,1,Afternoon,0


In [7]:
test_df[['Time', 'Hour', 'Day', 'Time_of_day', 'Class']].sample(n=5, random_state=42)

,Time,Hour,Day,Time_of_day,Class
24664,162488.0,21,1,Evening,0
37473,169034.0,22,1,Evening,0
37524,169064.0,22,1,Evening,0
24202,162295.0,21,1,Evening,0
38485,169701.0,23,1,Evening,0


In [8]:
train_df['Time_of_day'].value_counts()

Time_of_day
Morning      70643
Afternoon    53811
Evening      50312
Night        23842
Name: count, dtype: int64

In [9]:
val_df['Time_of_day'].value_counts()

Time_of_day
Afternoon    42310
Evening        249
Name: count, dtype: int64

In [10]:
test_df['Time_of_day'].value_counts()

Time_of_day
Evening    42559
Name: count, dtype: int64

In [11]:
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

px.histogram(train_df, x='amount_log', color='Time_of_day', title='Distribution of Log-Transformed Transaction Amounts', 
             labels={'amount_log': 'Log(Amount + 1)'}, 
             nbins=50,
             barmode='overlay')

In [12]:
frauds = train_df[train_df['Class'] == 1]
px.bar(frauds.groupby('Time_of_day').size().reset_index(name='count'), x='Time_of_day', y='count', title='Number of Fraudulent Transactions by Time of Day').show()

In [13]:
train_df.to_csv("../data/processed/model_train.csv", index=False)
val_df.to_csv("../data/processed/model_val.csv", index=False)
test_df.to_csv("../data/processed/model_test.csv", index=False)